# foco_queim_silver

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, trim
from pyspark.sql.types import DoubleType, StringType, IntegerType, TimestampType, NumericType

In [0]:
# ==============================
# WIDGETS / PARÂMETROS
# ==============================
dbutils.widgets.text("schema_bronze", "")
dbutils.widgets.text("table_bronze", "")
dbutils.widgets.text("schema_silver", "")
dbutils.widgets.text("table_silver", "")
dbutils.widgets.text("data_ref_carga", "")

schema_in = dbutils.widgets.get("schema_bronze").strip()
table_in = dbutils.widgets.get("table_bronze").strip()
schema_out = dbutils.widgets.get("schema_silver").strip()
table_out = dbutils.widgets.get("table_silver").strip()
data_ref_carga = dbutils.widgets.get("data_ref_carga").strip()

In [0]:
# ==============================
# TRATAMENTO DE ERROS: parâmetros obrigatórios
# ==============================
params = {
    "schema_in": schema_in,
    "table_in": table_in,
    "schema_out": schema_out,
    "table_out": table_out,
    "data_ref_carga": data_ref_carga
}
missing = [k for k, v in params.items() if v == ""]
if missing:
    raise ValueError(f"Parâmetros obrigatórios não informados: {', '.join(missing)}")

In [0]:
# ==============================
# Formatacao de tabela
# ==============================
spark = SparkSession.builder.getOrCreate()
tabela_bronze = f"{schema_in}.{table_in}"
tabela_silver = f"{schema_out}.{table_out}"

print(f"Lendo tabela bronze: {tabela_bronze} - partição data_ref_carga = {data_ref_carga}")

In [0]:
# ==============================
# LEITURA (filtrando por partição)
# ==============================
try:
    df = spark.read.table(tabela_bronze).filter(col("data_ref_carga") == lit(data_ref_carga))
except Exception as e:
    raise RuntimeError(f"Erro ao ler a tabela bronze {tabela_bronze}: {e}")

In [0]:
# ==============================
# TRATAMENTO DE NULOS
# ==============================

In [0]:
# detectar colunas por tipo
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]

# aplicar transformações
for c in string_cols:
    df = df.withColumn(c, when(col(c).isNull(), lit(" ")).otherwise(col(c)))

for c in numeric_cols:
    df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

In [0]:
# ============================================================
# CONVERSÕES PARA TIPOS NUMÉRICOS (com fallback = 0)
# ============================================================
df = (
    df
    .withColumn(
        "numero_dias_sem_chuva",
        when(trim(col("numero_dias_sem_chuva")) == "", lit(0))
        .otherwise(col("numero_dias_sem_chuva").cast(IntegerType()))
    )
    .withColumn(
        "precipitacao",
        when(trim(col("precipitacao")) == "", lit(0.0))
        .otherwise(col("precipitacao").cast(DoubleType()))
    )
    .withColumn(
        "risco_fogo",
        when(trim(col("risco_fogo")) == "", lit(0.0))
        .otherwise(col("risco_fogo").cast(DoubleType()))
    )
)

In [0]:
# ==============================
# REGRAS DE VALIDAÇÃO / CORREÇÃO
# ==============================
df = df.withColumn("risco_fogo", when(col("risco_fogo") == -999, lit(0)).otherwise(col("risco_fogo")))
df = df.withColumn("lat", when((col("lat") >= -90.0) & (col("lat") <= 90.0), col("lat")).otherwise(lit(0.0)))
df = df.withColumn("lon", when((col("lon") >= -180.0) & (col("lon") <= 180.0), col("lon")).otherwise(lit(0.0)))

In [0]:
# ==============================
# DEFINIÇÃO DOS TIPOS E COMENTÁRIOS PARA A TABELA SILVER
# ==============================
columns_definition = [
    ("id", "STRING", "Código único"),
    ("lat", "DOUBLE", "Latitude do centro do píxel de fogo ativo apresentada em unidade de graus decimais"),
    ("lon", "DOUBLE", "Longitude do centro do píxel de fogo ativo apresentada em unidade de graus decimais"),
    ("data_hora_gmt", "TIMESTAMP", "Horário de referência da passagem do satélite segundo o fuso horário de Greenwich (GMT). Formato: YYYY-MM-DDTHH:MM:SS.sss+00:00"),
    ("satelite", "STRING", "Nome do algoritmo utilizado e referência ao satélite provedor da imagem."),
    ("municipio", "STRING", "Nome do município. Para o Brasil foi utilizado como referência o dado do IBGE 2000."),
    ("estado", "STRING", "Nome do estado (nível 1 do GADM)."),
    ("pais", "STRING", "Nome do País (nível 0 do GADM)."),
    ("municipio_id", "INT", "Código/ID do município (referência IBGE quando aplicável)."),
    ("estado_id", "INT", "Código do estado"),
    ("pais_id", "INT", "Código do país"),
    ("numero_dias_sem_chuva", "INT", "Número de dias sem chuva até a detecção do foco."),
    ("precipitacao", "DOUBLE", "Valor da precipitação acumulada no dia até o momento da detecção do foco."),
    ("risco_fogo", "DOUBLE", "Valor do Risco de Fogo previsto para o dia da detecção do foco. Valores inválidos -999 foram setados para 0."),
    ("bioma", "STRING", "Nome do Bioma segundo referência do IBGE 2004. Para outros países o campo pode ficar nulo (representado como valor vazio)."),
    ("frp", "DOUBLE", "Fire Radiative Power, MW (megawatts)."),
    ("data_ref_carga", "STRING", "Data do processamento da partição (YYYY-MM-DD)")
]

# descrição geral da tabela
descricao_tabela = (
    "Tabela Silver - Focos de Queimadas e Incêndios (tratada)."
)

In [0]:
# ==============================
# CRIAR TABELA SE NÃO EXISTIR (COM OS COMENTÁRIOS) OU SOBRESCREVER DADOS SE EXISTIR
# ==============================
from pyspark.sql.utils import AnalysisException

def create_table_with_comments(table_fqdn, columns_def, partition_col, table_description):
    """
    Cria a tabela usando os tipos e comentários informados.
    Faz escape de aspas simples e caracteres problemáticos nos comentários para evitar SQL parse errors.
    """
    import re

    # Função utilitária para sanitizar comentários (remove caracteres problemáticos)
    def escape_comment_for_sql(text: str) -> str:
        if not text:
            return ""
        # substitui aspas simples por escape SQL padrão
        text = text.replace("'", "''")
        # remove aspas tipográficas e caracteres invisíveis que causam erro
        text = re.sub(r"[‘’´`]", "", text)
        # substitui quebras de linha por espaço
        text = text.replace("\n", " ").replace("\r", " ")
        # remove duplos espaços
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    cols_parts = []
    for name, dtype, comment in columns_def:
        safe_comment = escape_comment_for_sql(comment)
        cols_parts.append(f"{name} {dtype} COMMENT '{safe_comment}'")

    cols_sql = ",\n  ".join(cols_parts)

    sql = f"""
    CREATE TABLE {table_fqdn} (
      {cols_sql}
    )
    USING DELTA
    PARTITIONED BY ({partition_col})
    """

    # Criação da tabela
    spark.sql(sql)

    # Define a descrição geral da tabela
    safe_table_description = escape_comment_for_sql(table_description)
    spark.sql(f"ALTER TABLE {table_fqdn} SET TBLPROPERTIES (description = '{safe_table_description}')")


In [0]:
# verifica existência
table_exists = spark.catalog.tableExists(tabela_silver)  # uso interno catalog para suportar catalog.schema.table

if not table_exists:
    print(f"Tabela {tabela_silver} não existe. Criando com comentários...")
    try:
        create_table_with_comments(
            tabela_silver,
            columns_definition,
            "data_ref_carga",
            descricao_tabela
        )
    except Exception as e:
        raise RuntimeError(f"Erro ao criar tabela silver {tabela_silver}: {e}")
else:
    print(f"Tabela {tabela_silver} já existe. Irei gravar dados e atualizar comentários das colunas (se necessário).")

In [0]:
from pyspark.sql.functions import col, trim, when, lit
from pyspark.sql.types import NumericType, StringType, DateType, TimestampType

def sanitize_and_cast(df, tabela_destino: str):
    """
    Alinha DataFrame ao schema da tabela destino e remove valores inválidos.
    Garante tipos corretos ANTES da escrita Delta.
    """
    schema_destino = spark.table(tabela_destino).schema
    df_sel = df.select([c for c in df.columns if c in [f.name for f in schema_destino]])

    for field in schema_destino:
        nome = field.name
        tipo = field.dataType

        if nome not in df_sel.columns:
            continue

        # 🧹 1. Remove espaços e strings vazias
        df_sel = df_sel.withColumn(nome, trim(col(nome)))
        df_sel = df_sel.withColumn(nome, when((col(nome) == "") | (col(nome).isNull()), lit(None)).otherwise(col(nome)))

        # 🔢 2. Trata tipos numéricos
        if isinstance(tipo, NumericType):
            df_sel = df_sel.withColumn(
                nome,
                when(
                    col(nome).rlike("^[+-]?\\d+(\\.\\d+)?$"),
                    col(nome).cast(tipo.simpleString())
                ).otherwise(lit(None).cast(tipo.simpleString()))
            )

        # 🕓 3. Trata timestamps e datas
        elif isinstance(tipo, (TimestampType, DateType)):
            df_sel = df_sel.withColumn(nome, col(nome).cast(tipo.simpleString()))

        # 🔤 4. Strings apenas limpam espaços
        elif isinstance(tipo, StringType):
            df_sel = df_sel.withColumn(nome, when(col(nome) == "", lit(None)).otherwise(col(nome)))

        else:
            df_sel = df_sel.withColumn(nome, col(nome).cast(tipo.simpleString()))

    return df_sel


In [0]:
# Limpa e ajusta schema antes de gravar na tabela destino
df_sanitized = sanitize_and_cast(df, tabela_silver)

In [0]:
# ==============================
# 1️⃣ Função para reaplicar comentários
# ==============================
def apply_table_comments(table_fqdn: str, columns_def: list, table_description: str):
    """
    Aplica comentários em colunas e descrição da tabela no Delta.
    """
    def escape_comment(text):
        return (text or "").replace("'", "''").replace("\n", " ").replace("\r", " ")

    # Alterar comentário das colunas
    for name, _, comment in columns_def:
        safe_comment = escape_comment(comment)
        spark.sql(f"ALTER TABLE {table_fqdn} CHANGE COLUMN {name} COMMENT '{safe_comment}'")

    # Alterar descrição da tabela
    safe_table_description = escape_comment(table_description)
    spark.sql(f"ALTER TABLE {table_fqdn} SET TBLPROPERTIES (description = '{safe_table_description}')")

In [0]:
# ==============================
# GRAVANDO DADOS NA TABELA SILVER (por partição)
# - Usamos overwrite forçando apenas a partição específica para evitar perder histórico
# ==============================
try:
    # opção: sobrescrever apenas a partição específica (modo compatível com Delta)
    (
        df_sanitized.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "false")
        .partitionBy("data_ref_carga")
        .saveAsTable(tabela_silver)
    )
    print("Gravação realizada com sucesso na tabela silver.")
except Exception as e:
    raise RuntimeError(f"Erro ao gravar na tabela silver {tabela_silver}: {e}")

In [0]:
# ==============================
# 5️⃣ Reaplicar comentários e descrição
# ==============================
apply_table_comments(tabela_silver, columns_definition, descricao_tabela)
print("Comentários e descrição reaplicados ✅")

In [0]:
# ==============================
# FIM - resumo / contagem de registros
# ==============================
record_count = df.count()
print(f"Registros processados para data_ref_carga={data_ref_carga}: {record_count}")
print("Job finalizado com sucesso ✅")